In [1]:
import numpy as np
import pandas as pd

## expladatory data analysis

In [7]:
df = pd.read_csv("messy_food_waste.csv")

print(df.head())
print(df.shape) # (1560, 11)
print(df.columns)

# target = food_waste_kg

         date meals_served kitchen_staff  temperature_C  humidity_percent  \
0  2024-08-26           89             8          27.68             74.25   
1  2024-01-29          217             6          26.77             47.76   
2  2024-12-19          192             8            NaN             71.50   
3  2024-02-18          191             7            NaN             55.18   
4  2024-06-24          325            16          20.02             63.79   

   day_of_week  special_event  past_waste_kg staff_experience waste_category  \
0            0              0          46.36     Intermediate     Vegetables   
1            0              1          49.84           Expert     Vegetables   
2            3              0          39.53         Beginner           Meat   
3            6              1          27.12     Intermediate         Grains   
4            0              0            NaN         Beginner         GRAINS   

   food_waste_kg  
0          36.29  
1          42.22  

In [30]:
for x in df.columns:
    print(x, " = ", df[x].dtypes)
    # need to know type of columns

date  =  str
meals_served  =  int64
kitchen_staff  =  int64
temperature_C  =  float64
humidity_percent  =  float64
day_of_week  =  int64
special_event  =  int64
past_waste_kg  =  float64
staff_experience  =  str
waste_category  =  str
food_waste_kg  =  float64


In [32]:
miss = df.isnull().sum() / len(df) * 100 # how many percent of the feature is missing
print(miss)

duplicates = df.duplicated().sum()
print(duplicates) # 37 bor ekan

# meals_served        5.064103  str -> intga otqazishimiz kerak
# kitchen_staff       4.935897  str -> intga otqazish kerak
# temperature_C       7.884615  float64
# past_waste_kg       8.012821  float64
# staff_experience    7.884615  str -> format is different need to make everything lowercase and keyin normilize qisak boladi
# waste_category  =  str -> bu ham messy lowercase qilish kere
# not much we can fill everything

df = df.copy()

date                0.0
meals_served        0.0
kitchen_staff       0.0
temperature_C       0.0
humidity_percent    0.0
day_of_week         0.0
special_event       0.0
past_waste_kg       0.0
staff_experience    0.0
waste_category      0.0
food_waste_kg       0.0
dtype: float64
37


## Filling missing values

In [37]:
# kitchen_staff ni ichida: 10 staff yoki staff: 8 ga ohshaganlari bor ekan, shuni ichidan raqam chiqarib olamiz
df["kitchen_staff"] = (df["kitchen_staff"].astype(str).str.extract(r"(\d+(?:\.\d+)?)")[0])

# str to numeric(float)
df["meals_served"] = pd.to_numeric(df["meals_served"], errors="coerce")
df["kitchen_staff"] = pd.to_numeric(df["kitchen_staff"], errors="coerce")

# filling with meadian
df["meals_served"] = df["meals_served"].fillna(df["meals_served"].median())
df["kitchen_staff"] = df["kitchen_staff"].fillna(df["kitchen_staff"].median())

# now need to convert to int
# chunki odam 1.5 ta odam bololmidi
df["meals_served"] = df["meals_served"].astype(int)
df["kitchen_staff"] = df["kitchen_staff"].astype(int)

# filling temperature_C and past_waste_kg
df["temperature_C"] = df["temperature_C"].fillna(df["temperature_C"].median())
df["past_waste_kg"] = df["past_waste_kg"].fillna(df["past_waste_kg"].median())


# staff_experience make lowercase and fill with the mode
df["staff_experience"] = df["staff_experience"].str.lower()
df["staff_experience"] = df["staff_experience"].fillna(df["staff_experience"].mode()[0])

#waste_category  =  str -> bu ham messy lowercase qilish kere
df["waste_category"] = df["waste_category"].str.lower()

miss = df.isnull().sum() / len(df) * 100 # check one more time
print(miss) # 0 missing values

df = df.drop_duplicates() # drop duplicates
duplicates = df.duplicated().sum() # check one more time
print(duplicates) # 0 duplicates

df = df.copy()

date                0.0
meals_served        0.0
kitchen_staff       0.0
temperature_C       0.0
humidity_percent    0.0
day_of_week         0.0
special_event       0.0
past_waste_kg       0.0
staff_experience    0.0
waste_category      0.0
food_waste_kg       0.0
dtype: float64
0


In [42]:
df.shape # (1560, 11) - > (1523, 11)
df['day_of_week'].dtypes

dtype('int64')

## Outlierla ob tashlash

In [49]:
# 'date', 'meals_served', 'kitchen_staff', 'temperature_C',
#  'humidity_percent', 'day_of_week', 'special_event', 'past_waste_kg',
#  'staff_experience', 'waste_category', 'food_waste_kg'

# target food_waste_kg

numerical_columns = [
    'meals_served',
    'kitchen_staff',
    'temperature_C',
    'past_waste_kg',
    'food_waste_kg',
]

for x in numerical_columns:
    Q1 = df[x].quantile(0.25)
    Q3 = df[x].quantile(0.75)
    IQR = Q3 - Q1

    df = df[
        (df[x] >= Q1 - 1.5 * IQR) &
        (df[x] <= Q3 + 1.5 * IQR)
    ]

print(df.shape) # (1523, 11) -> (1391, 11)

df = df.copy()

(1391, 11)


## Feature engineering

In [53]:
# date ni togirlash kere
df["date"] = pd.to_datetime(df["date"], errors="coerce")

df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day

# how many meals each staff served
df["meals_per_staff"] = df["meals_served"] / df["kitchen_staff"]

df = df.copy()

## Encoding

In [52]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["experience_encoded"] = le.fit_transform(df["staff_experience"])


# one hot-encoding to waste_category
df = pd.get_dummies(
    df,
    columns=['waste_category'],
    dtype=int,
)

df = df.copy()

## Boglanishlar

In [51]:
corr = df.select_dtypes(include="number").corr()

print(corr["food_waste_kg"].sort_values(ascending=False))


df = df.copy()

food_waste_kg       1.000000
past_waste_kg       0.530318
meals_served        0.437177
meals_per_staff     0.240626
special_event       0.141361
humidity_percent    0.140388
temperature_C       0.055683
kitchen_staff       0.018584
day                 0.010911
day_of_week        -0.008132
month              -0.008151
Name: food_waste_kg, dtype: float64
